In [1]:
pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 2.5 MB/s eta 0:00:00


In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [12]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("new_foundry_api")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://myarchanafoundry.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [36]:
# 1. Define the input data
# In a real scenario, you can load this from a .txt or .json file
resume_data = """
RICHARD SANCHEZ
Education: Master of Business Management (2029-2030)
Work: Marketing Manager at Borcelle Studio (2030-Present)
Skills: Project Management, PR, Leadership...
"""

# 2. Use the Responses API for structured extraction
# Use the structured input pattern for GPT-4.1
# Updated call with 2026-compliant type naming
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": "Extract the following details into a JSON object: Name, Education, Latest Job, and Top 3 Skills. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",  # Changed from 'text'
                        "text": f"RESUME SOURCE:\n{resume_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Name": "Richard Sanchez",\n  "Education": "Master of Business Management (2029-2030)",\n  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",\n  "Top 3 Skills": ["Project Management", "PR", "Leadership"]\n}', type='output_text', logprobs=[])]


In [14]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the 'text' attribute of that item
    output_text_item = response.output[0].content[0]

    # In some SDK versions, it's a dict; in others, it's an object.
    # This handles both:
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Now parse the actual string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: DATA PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")
    # Troubleshooting: print the type to see what exactly was returned
    print(f"Actual content type: {type(response.output[0].content[0])}")

--- SUCCESS: DATA PARSED ---
{
  "Name": "Richard Sanchez",
  "Education": "Master of Business Management (2029-2030)",
  "Latest Job": "Marketing Manager at Borcelle Studio (2030-Present)",
  "Top 3 Skills": [
    "Project Management",
    "PR",
    "Leadership"
  ]
}


In [15]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=storagearchana;AccountKey=o5Twh1h+SJ0CQsGAk/VpR9WmgsXuZSohguOdKxEONKpobp6F+lagnKnFW5l5Qjg0nlbSGtUSpX0p+AStZOCtag==;EndpointSuffix=core.windows.net"

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "resume-output"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [16]:
import datetime

file_name = f"resume_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [17]:
blob_client = blob_service_client.get_blob_client(
    container="resume-output",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F672E47D939"',
 'last_modified': datetime.datetime(2026, 4, 21, 5, 31, 4, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'\xdb\xe3g\xe1$0\xd2G\xd7k\xff\x97\xbd\xe3\xdb`'),
 'client_request_id': '4a0a1f64-3d43-11f1-a888-0242ac1c000c',
 'request_id': 'f3120942-201e-004b-2050-d16094000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 5, 31, 3, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [18]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: resume_20260421_053049.txt


Build & Deploy an AI Research Paper Summarizer (Azure AI Foundry)
🎯 Objective
Students will:

Create AI resource

Build an AI agent

Deploy it

Call it using Python (Colab)

## Store summarized output in cloud

In [19]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("new2_f_api")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://myarchanafoundry.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [23]:
import json
import re

try:
    # 1. Get response content safely
    output_item = response.output[0].content[0]

    # 2. Handle both dict & object formats
    if isinstance(output_item, dict):
        raw_text = output_item.get("text", "")
    else:
        raw_text = output_item.text

    # 3. 🔥 Extract ONLY JSON (important fix)
    json_match = re.search(r"\{.*\}", raw_text, re.DOTALL)

    if not json_match:
        raise ValueError("No valid JSON found in response")

    json_string = json_match.group(0)

    # 4. Parse JSON
    extracted_data = json.loads(json_string)

    # 5. Validate expected fields (research paper structure)
    expected_keys = [
        "title",
        "abstract_summary",
        "key_points",
        "methodology",
        "conclusion",
        "limitations"
    ]

    missing_keys = [key for key in expected_keys if key not in extracted_data]

    print("📄 --- SUCCESS: SUMMARIZED DATA ---")
    print(json.dumps(extracted_data, indent=2))

    # 6. Warning if something missing
    if missing_keys:
        print(f"⚠️ Missing keys: {missing_keys}")

except Exception as e:
    print(f"❌ Parsing Error: {e}")

    # Debug info
    try:
        print("🔍 Raw Output:")
        print(raw_text)
    except:
        print("⚠️ Could not print raw output")

📄 --- SUCCESS: SUMMARIZED DATA ---
{
  "title": "Enhancing Student Learning with AI-Based Personalized Education Systems",
  "abstract_summary": "This paper examines the transformative impact of Artificial Intelligence (AI) on education, focusing on AI-driven personalized education systems. It analyzes how AI technologies tailor instruction to individual student needs, discusses improvements in engagement and learning outcomes, and evaluates challenges and future directions for implementation.",
  "key_points": [
    "AI-based education systems adapt curriculum and pace to individual student abilities.",
    "Personalized learning through AI leads to improved student engagement and performance.",
    "AI systems provide real-time feedback, helping educators identify and address learning gaps.",
    "Key challenges include data privacy, teacher adaptation, and resource allocation."
  ],
  "methodology": "The paper conducts a comprehensive literature review of recent studies (2017\u20132

In [24]:
from azure.storage.blob import BlobServiceClient
import json

# 🔹 Azure Storage Connection String
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=storagearchana;AccountKey=o5Twh1h+SJ0CQsGAk/VpR9WmgsXuZSohguOdKxEONKpobp6F+lagnKnFW5l5Qjg0nlbSGtUSpX0p+AStZOCtag==;EndpointSuffix=core.windows.net"

# 🔹 Create Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

# 🔹 Container Name (updated)
container_name = "research-summaries"

# 🔹 Your summarized JSON (previous step se aayega)
# Example:
# extracted_data = {...}

# Convert JSON → string
json_data = json.dumps(extracted_data, indent=2)

# 🔹 Blob name (dynamic rakhna better hai)
blob_name = "paper_summary_1.json"

# 🔹 Create blob client
blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

# 🔹 Upload to Azure
blob_client.upload_blob(json_data, overwrite=True)

print("✅ Research summary uploaded successfully!")

✅ Research summary uploaded successfully!


In [25]:
import datetime

# 🔹 Better file naming for research summaries
file_name = f"research_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

In [28]:
blob_client = blob_service_client.get_blob_client(
    container="research-summaries",   # ✅ updated container name
    blob=file_name
)

# 🔹 Use parsed JSON (recommended)
import json

data_to_upload = json.dumps(extracted_data, indent=2)

blob_client.upload_blob(data_to_upload, overwrite=True)

print("✅ Summary uploaded successfully!")

✅ Summary uploaded successfully!


In [29]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: research_summary_20260421_062433.json


Build & Deploy an AI Medical Report Interpreter (Azure AI Foundry)
🎯 Objective
Students will:

Create AI resource

Build an AI agent

Deploy it

Call it using Python (Colab)

Store interpreted reports in cloud

In [30]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("new2_f_api")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://myarchanafoundry.openai.azure.com/openai/v1"
)

# Note: api_version is usually NOT required when using this path in 2026

In [37]:
# 1. Define the input data (Medical Report)
report_data = """
Patient Name: John Doe
Hemoglobin: 9 g/dL (Low)
WBC Count: 12000 /µL (High)
Platelets: Normal
Glucose: 180 mg/dL (High)

Notes: Patient shows fatigue and mild fever.
"""

# 2. Call Responses API for medical interpretation
try:
    response = client.responses.create(
        model="YOUR_DEPLOYMENT_NAME",   # 🔴 IMPORTANT (Azure deployment name)
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
Analyze the following medical report and return structured JSON with:

- patient_summary
- key_findings (bullet points)
- abnormal_values
- possible_conditions (no diagnosis, just indications)
- recommendations
- severity (Low / Moderate / High)

Return ONLY valid JSON.
"""
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{report_data}"
                    }
                ]
            }
        ]
    )

    print("🏥 --- RAW OUTPUT ---")

    # 🔹 Safe extraction
    output_item = response.output[0].content[0]

    if isinstance(output_item, dict):
        output_text = output_item.get("text", "")
    else:
        output_text = output_item.text

    print(output_text)

except Exception as e:
    print(f"❌ Extraction Error: {e}")

❌ Extraction Error: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}
